CMS compiles claims data for Medicare and Medicaid patients across a variety of categories and years. This includes Inpatient and Outpatient claims, Master Beneficiary Summary Files, and many other files. Indicators from this data source have been computed by personnel in CDC's Division for Heart Disease and Stroke Prevention (DHDSP).

The system is designed to integrate multiple indicators from many data sources to provide a comprehensive picture of the public health burden of CVDs and associated risk factors in the United States. The data are organized by location (national and state) and indicator. The data can be plotted as trends and stratified by sex and race/ethnicity.

# **Importing necessary libraries**

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data= pd.read_csv('/kaggle/input/heart-disease-and-stroke-prevention/heart_disease_data.csv')

# **Data Summarization**

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
data.describe().T

# **Value counts:**

In [ ]:
# Value counts of location description
data.value_counts(['LocationDesc'])

Equal number of data is collected from each location in US.

In [ ]:
data.value_counts(['Year'])

Same number of dataset(4264) is collected from 2004 to 2013

In [ ]:
data.value_counts(['Topic'])

6 Topics are present in the Dataset.

In [ ]:
data.value_counts(['Indicator'])

In [ ]:
data.nunique()

In [ ]:
data.isnull().sum()

# #**Exploratory Data Analysis**

# **Univariate Analysis**

# **1.Pie chart of different categories of people taken for survey.**

In [ ]:
review_percent1 = data.Break_Out_Category.value_counts().reset_index()
review_percent1

In [ ]:
# # Plotting pie chart of break_out_category
# review_percent = data.Break_Out_Category.value_counts().reset_index()
# plt.figure(figsize=(6,4))
# plt.pie(review_percent['count'],labels=	Break_Out_Category, autopct='%1.1f%%', startanagle=140)
# plt.axis('equal')
# plt.show

# Plotting pie chart of break_out_category
review_percent = data.Break_Out_Category.value_counts().reset_index()
plt.figure(figsize=(6, 4))
plt.pie(review_percent['count'], labels=review_percent['Break_Out_Category'], autopct='%1.1f%%', startangle=140)
plt.axis('equal')
plt.show()



Break out category is divided into Age, Gender, Race and Overall and which is again divided in to the different categories.

In [ ]:
fig , axes=plt.subplots(nrows=2, ncols=1,figsize=(12,10))

sns.countplot(data=data, x="Break_Out_Category",ax=axes[0])
sns.countplot(data=data, x="Break_Out",ax=axes[1])

# **2.Histogram of Topic**

In [ ]:
# Histplot of Topic to find out the count
sns.histplot(y="Topic",data=data,color="darkblue",edgecolor='black')
plt.title('Histogram of Topic')

***This graph shows the health conditions of US residents.***

# **From this graph, it is clear that Heart Failure is very high as compared to others .**

# **3.Frequency graph of dataset**

In [ ]:
# Frequency graph of dataset
freqgraph=data.select_dtypes(include=['float64','int64'])
freqgraph.hist(figsize=(10,8))
plt.tight_layout()
plt.show()

**4.Plot of Priority Areas**

In [ ]:
data

In [ ]:
#PriorityAreas
fig , axes=plt.subplots(nrows=4 , ncols=1,figsize=(10,8))

data['PriorityArea1'].fillna('Missing', inplace=True)
data['PriorityArea2'].fillna('Missing', inplace=True)
data['PriorityArea3'].fillna('Missing', inplace=True)
data['PriorityArea4'].fillna('Missing', inplace=True)

sns.countplot(data=data, x="PriorityArea1",ax=axes[0])
sns.countplot(data=data, x="PriorityArea2",ax=axes[1])
sns.countplot(data=data, x="PriorityArea3",ax=axes[2])
sns.countplot(data=data, x="PriorityArea4",ax=axes[3])
plt.show()

**PriorityArea2,PriorityArea4 contains None values throughout, also these variables dont serve any purpose in prediction of our target variable.Let us drop those variables.**

# **5.Plot of DataSource,Category and Data value type**

In [ ]:
fig , axes=plt.subplots(nrows=3, ncols=1,figsize=(12,10))
sns.countplot(data=data, x="DataSource",ax=axes[0])
sns.countplot(data=data, x="Category",ax=axes[1])
sns.countplot(data=data, x="Data_Value_Type",ax=axes[2])

**Data source have medicare values throughout also category column consists of cardiovascular diseases values entirely.**

**This is not a variable feature in our dataset ,it is observed that it is constant throughout. Hence dropping them wil not affect our prediction.**

# **Multivariate Analysis**

# **1.Correlation Heatmap**

In [ ]:
# Identify non-numeric columns
non_numeric_columns = data.select_dtypes(exclude=['number']).columns.tolist()

# Exclude non-numeric columns when computing the correlation matrix
correlation_matrix = data.drop(columns=non_numeric_columns).corr()

correlation_matrix

In [ ]:
plt.figure(figsize=(8,8))
sns.heatmap(correlation_matrix,annot=True,cmap='GnBu')
plt.show()

**From above correlation it is clear that Data_value and Data_Value_Alt are similar.** 

**Also high correlation is seen between Data_value,HighConfidenceLimit and LowConfidenceLimit**

# **2. Pairplot**

In [ ]:
sns.pairplot(data)

# **Data Cleaning**

# **1. Missing values**

In [ ]:
# Checking Duplicate Values
print(len(data[data.duplicated()]))

In [ ]:
#Checking for null values
data.isna().sum()

In [ ]:
#Missing Values
import missingno as msno
msno.matrix(data,figsize=(12,12))

There are total 86629 missing values in the given dataset.

From this, 529 Missing values are in Datavalue variabe.

42111 missing values each for Data value foot note symbol and Datavalue foot note. This can affect the performance of our model.

529 missing values each for confidence limit low and confidence limit high respectvely.

820 missing values for Geolocation.

In [ ]:
#Columns: Data_Value_Footnote_Symbol,Data_Value_Footnote consists tremendous number of missing values and are not required for prediction.
#It is adviseable to drop them.Also GeoLocation is to be dropped as it is unnecessary for prediction
#Dropping Data_Value
data.drop('Data_Value_Footnote_Symbol', axis = 1,inplace=True)
data.drop('Data_Value_Footnote', axis = 1,inplace=True)
data.drop('GeoLocation', axis = 1,inplace=True)
data.drop('Data_Value', axis = 1,inplace=True)

In [ ]:
from scipy import stats
for col in data.describe().columns:
  plt.figure(figsize=(12,3))

# Skewness Distribution
  plt.subplot(131)
  sns.distplot(data[col], label="skew" )
  plt.legend()

# Boxplot - For outliers detection
  plt.subplot(132)
  sns.boxplot(data[col])
  plt.show()

From above plot it can be observed that, year and LocationID is free from outliers where rest of the numeric features poses outliers.

Also these features are right skewed. Let us see the number of null values present in them.

In [ ]:
# Filling the missing values using median(Right skewed data)

filler = data["HighConfidenceLimit"].median()
data["HighConfidenceLimit"] = data["HighConfidenceLimit"].fillna(filler)

filler = data["LowConfidenceLimit"].median()
data["LowConfidenceLimit"] = data["LowConfidenceLimit"].fillna(filler)

In [ ]:
# Recheck for null values after Filling and Dropping
data.isna().sum()

**2.Removing Outliers**

In [ ]:
# Removing outliers using interquantile range
#Defining function as we have to remove outliers from multiple columns

def outliers(data,ft):
  Q1 = data[ft].quantile(0.25)
  Q3= data[ft].quantile(0.75)
  IQR =Q3 - Q1
  lower_bound=Q1-1.5*IQR
  upper_bound=Q3+1.5*IQR

#List to store the index values of the outliers
  ol_list = data.index[(data[ft]<lower_bound) | (data[ft]>upper_bound)]
  return ol_list

In [ ]:
#Create an empty list to store the output indices from multiple columns
index_list=[]
for feature in ['Data_Value_Alt','LowConfidenceLimit','HighConfidenceLimit']:
  index_list.extend(outliers(data,feature))

In [ ]:
print(index_list)

In [ ]:
#Defining remove function to get cleaned dataframe without outliers
def remove(data,ol_list):
  ol_list=sorted(set(ol_list))                          #To get the sorted list
  data=data.drop(ol_list)
  return data

In [ ]:
#Storing the cleaned data into new data frame named df1
data1=remove(data,index_list)

In [ ]:
#New shape of cleaned data
data1.shape

In [ ]:
#Lets verify the removal of outliers graphically
#Visualization

from scipy import stats
for col in data1.describe().columns:
  plt.figure(figsize=(12,3))

# Skewness Distribution
  plt.subplot(131)
  sns.distplot(data1[col], label="skew" )
  plt.legend()

# Boxplot - For outliers detection
  plt.subplot(132)
  sns.boxplot(data1[col])
  plt.show()

# **Cleaning of categorical features**

In [ ]:
compare_location = pd.DataFrame({'LocationAbbr': data1.LocationAbbr, 'LocationDesc': data1.LocationDesc})
compare_location.value_counts()

From above we can observe that LocationAbbr is the abbrevation of LocationDesc.

We can drop these columns as they are unnecessary for prediction.

In [ ]:
compare_topic = pd.DataFrame({'Topic': data1.Topic,'TopicId': data1.TopicId})
                             
compare_topic.value_counts()

In [ ]:
compare_breakout = pd.DataFrame({'Break_Out': data1.Break_Out,
                             'BreakOutId': data1.BreakOutId})
compare_breakout.value_counts()

In [ ]:
compare_breakout_cat = pd.DataFrame({'Break_Out_Category': data1.Break_Out_Category,
                           'BreakOutCategoryId': data1.BreakOutCategoryId})
compare_breakout_cat.value_counts()

In [ ]:
compare_area = pd.DataFrame({'PriorityArea1': data1.PriorityArea1,
                           'PriorityArea3': data1.PriorityArea3,
                           'Data_Value_Unit': data1.Data_Value_Unit})
compare_area.value_counts()

In [ ]:
#Dropping unwanted columns
b=['PriorityArea2', 'PriorityArea4','DataSource','Category','Data_Value_Type','LocationAbbr','LocationDesc','TopicId','BreakOutId','BreakOutCategoryId','CategoryId','LocationID','Data_Value_TypeID','IndicatorID']
for i in np.arange(len(b)):
    data1.drop(b[i], axis=1,inplace=True)

In [ ]:
data1.columns

In [ ]:
data1.shape

# **Feature Engineering for Priority Area variables**

In [ ]:
#import label encoder
from sklearn.preprocessing import LabelEncoder
#creating an instance LabelEncoder
label_en =LabelEncoder()

In [ ]:
#Create dummies for PriorityArea1
p1=pd.get_dummies(data1['PriorityArea1'])

#Create dummies for PriorityArea3
p3=pd.get_dummies(data1['PriorityArea3'])

#Concatenating PriorityAreas
data1=pd.concat([data1,p1,p3],axis=1)

In [ ]:
data1

In [ ]:
print(data1.dtypes)

In [ ]:
data1["Million Hearts"]=data1["Million Hearts"]*data1["PriorityArea1"]
data1["Healthy People 2020"]=data1["Healthy People 2020"]*data1["PriorityArea3"]

In [ ]:
data1.head()

In [ ]:
#create new column "PriorityArea"
data1["PriorityArea"]=data1["Million Hearts"]+data1["Healthy People 2020"]

In [ ]:
data1

In [ ]:
#removing unwanted columns
# data1=data1.drop(['None'],axis=1)

In [ ]:
#removing unwanted columns
data1=data1.drop(["Million Hearts","Healthy People 2020","PriorityArea1","PriorityArea3"],axis=1)

In [ ]:
#value counts of newcolumn
data1["PriorityArea"].value_counts()

In [ ]:
#label encoding new PriorityArea
data1["PriorityArea"]=label_en.fit_transform(data1["PriorityArea"])

In [ ]:
#value counts of newcolumn
data1["PriorityArea"].value_counts()

In [ ]:
data1.head()

In [ ]:
data1.shape

In [ ]:
data1.columns

In [ ]:
data1.info()

**Label Encoding**

In [ ]:
data1['Topic']=label_en.fit_transform(data1['Topic'])
data1['Indicator']=label_en.fit_transform(data1['Indicator'])
data1['Data_Value_Unit']=label_en.fit_transform(data1['Data_Value_Unit'])
data1['Break_Out_Category']=label_en.fit_transform(data1['Break_Out_Category'])
data1['Break_Out']=label_en.fit_transform(data1['Break_Out'])

In [ ]:
data1.info()

In [ ]:
#Splitting Dataset
y=data1['Topic']
X=data1.drop('Topic',axis=1)

#**Model Building**

In [ ]:
#Importing necessary libraries for model buiding

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Splitting the data into training set and testset
from sklearn.model_selection import train_test_split
X_train1, X_test1, y_train1, y_test1 = train_test_split(X,y, test_size = 0.2, random_state = 100)
print("Size of training set:", X_train1.shape)
print("Size of test set:", X_test1.shape)

**1.Multinomial Logistic Regression**

In [ ]:
lr1=LogisticRegression()
parameters = [{'penalty':['l1','l2','elasticNet']},{'multi_class':['multinomial']},
              {'C':[1, 10, 100, 1000]}]
grid_lr1 = GridSearchCV(estimator =lr1,
                           param_grid = parameters,
                           scoring = 'accuracy',
                           cv = 5,
                           verbose=0)
grid_lr1.fit(X_train1,y_train1)

In [ ]:
y_pred_train1=grid_lr1.predict(X_train1)
y_pred_test1=grid_lr1.predict(X_test1)

In [ ]:
#Classification report of train data
print(classification_report(y_pred_train1,y_train1))

In [ ]:
#Classification report of test data
print(classification_report(y_pred_test1,y_test1))

In [ ]:
#Accuracy Score
ascore_lr_train1=accuracy_score(y_pred_train1,y_train1)
ascore_lr_train1

In [ ]:
#Accuracy Score
ascore_lr1=accuracy_score(y_pred_test1,y_test1)
ascore_lr1

**2.Decision Tree**

In [ ]:
d_tree1 = DecisionTreeClassifier()
grid_dtc1 = GridSearchCV(d_tree1, param_grid = {'max_depth': (5, 30), 'max_leaf_nodes': (10, 100)}, scoring = 'accuracy', cv = 7)
grid_dtc1.fit(X_train1, y_train1)

In [ ]:
y_pred_train_dtg1 = grid_dtc1.predict(X_train1)
y_pred_test_dtg1 = grid_dtc1.predict(X_test1)

In [ ]:
#Classification report for train data
print(classification_report(y_train1,y_pred_train_dtg1))

In [ ]:
#Classification report for test data
print(classification_report(y_test1,y_pred_test_dtg1))

In [ ]:
#Accuracy Score
ascore_dtg_train1=accuracy_score(y_pred_test_dtg1,y_test1)
ascore_dtg_train1

In [ ]:
#Accuracy Score
ascore_dtg1=accuracy_score(y_pred_test_dtg1,y_test1)
ascore_dtg1

**3.KNN**

In [ ]:
knn1 = KNeighborsClassifier()
k_range = list(range(1,41))
param_grid = dict(n_neighbors=k_range)

# defining parameter range
knn_grid1 = GridSearchCV(knn1, param_grid, cv=7, scoring='accuracy', return_train_score=False,verbose=1)

# fitting the model for grid search
knn_grid1.fit(X_train1, y_train1)

In [ ]:
y_pred_train_knn1=knn_grid1.predict(X_train1)
y_pred_test_knn1=knn_grid1.predict(X_test1)

In [ ]:
#Classification report for train data
print(classification_report(y_train1,y_pred_train_knn1))

In [ ]:
#Classification report for test data
print(classification_report(y_test1,y_pred_test_knn1))

In [ ]:
#Accuracy Score
ascore_knn_train1=accuracy_score(y_pred_train_knn1,y_train1)
ascore_knn_train1

In [ ]:
#Accuracy Score
ascore_knn1=accuracy_score(y_pred_test_knn1,y_test1)
ascore_knn1

In [ ]:
model_list1 = ['Logistic regression','Decision Tree','KNN']
train_result_list1 = [ascore_lr_train1*100,ascore_dtg_train1*100,ascore_knn_train1*100]
result_list1 = [ascore_lr1*100,ascore_dtg1*100,ascore_knn1*100]

df_result1 = pd.DataFrame()
df_result1['Model name'] = model_list1
df_result1['Accuracy Score (Train)'] = train_result_list1
df_result1['Accuracy Score(Test)'] = result_list1

df_result1

#**Principal Component Analysis**

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaler.fit(X)

In [ ]:
scaled_data=scaler.transform(X)

In [ ]:
scaled_data

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca=PCA(n_components=2)

In [ ]:
pca.fit(scaled_data)

In [ ]:
x_pca=pca.transform(scaled_data)

In [ ]:
print('scaled_data shape',scaled_data.shape)
print('x_pca shape',x_pca.shape)

In [ ]:
x_pca

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(x_pca[:,0],x_pca[:,1],c=y)
plt.xlabel('First principle component')
plt.ylabel('Second principle component')

#**Train Test Split**

In [ ]:
#Train-Test Split after applying PCA
X_train, X_test, y_train, y_test = train_test_split(x_pca,y, test_size = 0.2, random_state = 100)
print("Size of training set:", X_train.shape)
print("Size of test set:", X_test.shape)

#**Model Building after PCA application**

**1.Logistic Regression**

In [ ]:
lr=LogisticRegression()
parameters = [{'penalty':['l1','l2','elasticNet']},{'multi_class':['multinomial']},
              {'C':[1, 10, 100, 1000]}]
grid_lr = GridSearchCV(estimator =lr,
                           param_grid = parameters,
                           scoring = 'accuracy',
                           cv = 5,
                           verbose=0)
grid_lr.fit(X_train,y_train)

In [ ]:
y_pred_train=grid_lr.predict(X_train)
y_pred_test=grid_lr.predict(X_test)

In [ ]:
#Classification report of train data
print(classification_report(y_pred_train,y_train))

In [ ]:
#Classification report of test data
print(classification_report(y_pred_test,y_test))

In [ ]:
#Accuracy Score
ascore_lr_train=accuracy_score(y_pred_train,y_train)
ascore_lr_train

In [ ]:
#Accuracy Score
ascore_lr=accuracy_score(y_pred_test,y_test)
ascore_lr

**2.KNN Classifier**

In [ ]:
knn = KNeighborsClassifier()
k_range = list(range(1,41))
param_grid = dict(n_neighbors=k_range)

# defining parameter range
knn_grid = GridSearchCV(knn, param_grid, cv=7, scoring='accuracy', return_train_score=False,verbose=1)

# fitting the model for grid search
knn_grid.fit(X_train, y_train)

In [ ]:
y_pred_train_knn=knn_grid.predict(X_train)
y_pred_test_knn=knn_grid.predict(X_test)

In [ ]:
#Classification report for train data
print(classification_report(y_train,y_pred_train_knn))

In [ ]:
#Classification report for test data
print(classification_report(y_test,y_pred_test_knn))

In [ ]:
#Accuracy Score
ascore_knn_train=accuracy_score(y_pred_train_knn,y_train)
ascore_knn_train

In [ ]:
#Accuracy Score
ascore_knn=accuracy_score(y_pred_test_knn,y_test)
ascore_knn

**3.Decision Tree Classifier**

In [ ]:
d_tree = DecisionTreeClassifier(max_depth=5)
d_tree.fit(X_train,y_train)

In [ ]:
y_pred_train_dt=d_tree.predict(X_train)
y_pred_test_dt=d_tree.predict(X_test)

In [ ]:
#Classification report for train data
print(classification_report(y_train,y_pred_train_dt))

In [ ]:
#Classification report for test data
print(classification_report(y_test,y_pred_test_dt))

In [ ]:
#Accuracy Score
ascore_dt_train=accuracy_score(y_pred_train_dt,y_train)
ascore_dt_train

In [ ]:
#Accuracy Score
ascore_dt=accuracy_score(y_pred_test_dt,y_test)
ascore_dt

**4.XGBoost Classifier**

In [ ]:
params = {
            'objective':'binary:logistic',
            'max_depth': 4,
            'alpha': 10,
            'learning_rate': 1.0,
            'n_estimators':100
        }
xgb_clf= XGBClassifier(**params)
xgb_clf.fit(X_train, y_train)

In [ ]:
print(xgb_clf)

In [ ]:
y_pred_train_xgb=xgb_clf.predict(X_train)
y_pred_test_xgb= xgb_clf.predict(X_test)

In [ ]:
#Classification report for train data
print(classification_report(y_train,y_pred_train_xgb))

In [ ]:
#Classification report for test data
print(classification_report(y_test,y_pred_test_xgb))

In [ ]:
#Accuracy Score
ascore_xgb_train=accuracy_score(y_pred_train_xgb,y_train)
ascore_xgb_train

In [ ]:
#Accuracy Score
ascore_xgb=accuracy_score(y_pred_test_xgb,y_test)
ascore_xgb

#**Result**

In [ ]:
model_list = ['Logistic regression','KNN','Decision Tree','XGBoost']
train_result_list = [ascore_lr_train*100,ascore_knn_train*100,ascore_dt_train*100,ascore_xgb_train*100]
result_list = [ascore_lr*100,ascore_knn*100,ascore_dt*100,ascore_xgb*100]


df_result = pd.DataFrame()
df_result['Model name'] = model_list
df_result['Accuracy Score (Train)'] = train_result_list
df_result['Accuracy Score(Test)'] = result_list

df_result

In [ ]:
#Bar graph of accuracy score of models perfromed
models = ['Logistic Reg', 'KNN','Decision Tree','XGBoost']
acc_scores=[ascore_lr,ascore_knn,ascore_dt,ascore_xgb]
plt.bar(models, acc_scores, color=['pink', 'green', 'lightblue','yellow'])
plt.ylabel("Accuracy scores")
plt.title("Model Accuracy")
plt.show()

#**Conclusion**

After passing through different models, accuracy of KNN is high as compared to others. Therefore, KNN is choosen as the best model for predicting heart disease dataset.

**Verification**


**Topic :**

0:Major Cardiovascular Disease                     

1:Stroke                                           

2:Diseases of the Heart (Heart Disease)            

3:Heart Failure                                    

4:Acute Myocardial Infarction (Heart Attack)       

5:Coronary Heart Disease                           


**Indicator :**

0 : Prevalence of all heart disease hospitalizations among all hospitalizations, US Medicare FFS beneficiaries (65+)

1 : Prevalence of heart failure hospitalizations among all hospitalizations, US Medicare FFS beneficiaries (65+)  
         
                                
2 : Prevalence of cerebrovascular disease hospitalizations among all hospitalizations, US Medicare FFS beneficiaries(65+)


3 : Prevalence of major cardiovascular disease hospitalizations among all hospitalizations, US Medicare FFS beneficiaries(65+)

4 : Prevalence of coronary heart disease hospitalizations among all hospitalizations, US Medicare FFS beneficiaries (65+)

5 : Prevalence of heart attack hospitalizations among all hospitalizations, US Medicare FFS beneficiaries(65+)

6 : Rate of hospitalizations among older adults with heart failure as the principal diagnosis (among FFS Medicare beneficiaries(65+)                  
        
7 : Rate of hospitalizations among adults aged 75 to 84 years with heart failure as the principal diagnosis (among FFS Medicare beneficiaries(65+)

8 : Rate of hospitalizations among adults aged 85 years and older with heart failure as the principal diagnosis (among FFS Medicare beneficiaries (65+))    

9 : Rate of hospitalizations among adults aged 65 to 74 years with heart failure as the principal diagnosis (among FFS Medicare beneficiaries (65+))



**Data_Value_Unit :**

1 : Percent (%)       
2 : Rate per 1,000    


**Break_Out_Category :**

0 : Age

1 : Gender          
2 : Overall     
3 : Race


**Break_Out :**

0 : 65+                   
1 : 75+

2 : Other

3 : Hispanic  
4 : Non-Hispanic Black  
5 : Non-Hispanic White             
6 : Female
7 : Male                 
8 : Overall               
   
                 
              
    
                   


In [ ]:
print("Enter details for prediction")
a = int(input("Year: "))
b = float(input("Indicator: "))
c = float(input("Data_Value_Unit: "))
d = float(input("Data_Value_Alt: "))
e = float(input("LowConfidenceLimit:"))
f = float(input("HighConfidenceLimit: "))
g = float(input("Break_Out_Category: "))
h = float(input("Break_Out: "))
i = float(input("PriorityArea: "))


features = [[a, b, c, d, e, f,g,h,i]]
scaled_data=scaler.transform(features)
x_pca=pca.transform(scaled_data)


print("========================================================================================")

print("Prediction : ", knn_grid.predict(x_pca))

o=knn_grid.predict(x_pca)

def disease_prediction (o):

  s=''

  if o ==0:
    s='Major Cardiovascular Disease'
  elif o ==1:
    s='Stroke'
  elif o ==2:
    s='Diseases of the Heart (Heart Disease)'
  elif o ==3:
    s='Heart Failure'
  elif o ==4:
    s='Acute Myocardial Infarction (Heart Attack)'
  elif o ==5:
    s='Coronary Heart Disease'
  else:
    s='Invalid Details'
  return s

print(disease_prediction (o))